## Process results from tree logs

In [ ]:
import os
import tqdm
from datasets import Dataset

from agents.roles.evaluator import Evaluator
from utils.utils import process_tree_logs, generate_final_answer


# Process tree logs
logs_path = input("Enter the path to the tree logs: ")
jsonl_files = [f for f in os.listdir(logs_path) if f.endswith('.jsonl')]

online_model_kwargs = {
        'model_name': 'openai/qwen3-8B', 
        'url': 'http://ip-10-4-226-205:30000/v1', 
        'api_key': 'your_api_key_here',  # Replace with your actual API key
        'client_type': 'openai',  # Use 'litellm' for LiteLLMClient or 'openai' for OpenAIClient
        'concurrency': 64,
    }
eval_kwargs = {
        # For creative tasks (creative writing) set it ~ 1, 
        # For logical or factual tasks (summarization, coding, analysis) set it ~ 0
        # For general conversation set it ~ 0.7
        'temperature': 0.1,  
        'n': 5, 
        'top_p': 0.9,
        'max_tokens': 1024*8,  # Set to a high value to allow for long responses
        # Want more varied responses (alongside high temperature) set top_k to 50 - 100 
        # For greedy decoding set it to 1
        'top_k': 20,
        'tensor_parallel_size': 1,
        'reasoning_effort': 'medium',  # Set to 'high'/'medium'/'low' for using thinking capabilities
    }
evaluator = Evaluator(
    client_kwargs=online_model_kwargs, 
    generate_kwargs=eval_kwargs, 
    # verbose=True,
    use_cache=True, 
    cache_dir="mcts_cache/evaluator_cache",
)

data = []
for file in tqdm.tqdm(jsonl_files, desc="Processing JSONL files"):
    # Add logic to process idx from file name if needed
    full_path = os.path.join(logs_path, file)
    user_question, all_answers = process_tree_logs(full_path)
    data.append({
        'id': file,
        'question': user_question,
        'answers': all_answers,
    })
dataset = Dataset.from_list(data)

dataset = dataset.map(
    lambda example: generate_final_answer(example, evaluator),
    # num_proc=512,  # Adjust based on your system's capabilities
    desc="Generating final answers",
    remove_columns=dataset.column_names,
    num_proc=512
)

